# Ноутбук 11 — Эксперимент 8: Подбор гиперпараметров через Optuna

**Назначение.** Honest hyperparameter tuning через Optuna с вложенной
кросс-валидацией для трёх ключевых моделей работы:
1. **XGBoost block** — главный результат диплома (раздел 3.4).
2. **Random Forest window** — победитель оконной задачи (раздел 3.3).
3. **LSTM window** — для честного сравнения с classical моделями (раздел 3.6.2).

**Стратегия.**
- Внешний цикл: LOSO (32 фолда) — даёт финальные метрики.
- Внутренний цикл: 3-fold по train-субъектам — для Optuna.
- Каждое внутреннее исследование: 20 trial (XGB), 20 (RF), 15 (LSTM).
- Sampler: TPE (Tree-structured Parzen Estimator), MedianPruner.
- Целевая метрика: F1-macro.

**Время выполнения.** ~2-3 ч на GPU (RTX 5070 Ti).

**Защита от утечки.** Optuna запускается ТОЛЬКО на train-субъектах
внешнего LOSO-фолда. Тестовый субъект не используется ни на одном этапе
выбора гиперпараметров.


In [1]:
import sys, time, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

NB_ROOT = Path.cwd()
if str(NB_ROOT) not in sys.path:
    sys.path.insert(0, str(NB_ROOT))

import numpy as np
import pandas as pd
import optuna
import torch
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score
from xgboost import XGBClassifier

from modules import config
from modules.experiments import (
    NON_FEATURE_COLS, add_per_subject_target, per_subject_zscore,
    build_block_level_dataset,
)
from modules.validation import (
    compute_classification_metrics, subject_independent_splits,
)
from modules.deep import build_sequences, K_DEFAULT, set_seed

R = config.RESULTS_DIR
SEED = config.RANDOM_SEED
optuna.logging.set_verbosity(optuna.logging.WARNING)
print(f"Results dir: {R}")


Results dir: D:\Programming\Python\Diplom\source\results


## 1. Загрузка данных

In [2]:
df = pd.read_csv(R / "feature_table.csv")
df = add_per_subject_target(df)
ALL = [c for c in df.columns if c not in NON_FEATURE_COLS and c != "arousal_class_persubj"]
print(f"shape: {df.shape}, features: {len(ALL)}")

# Window-level
y_w = df["arousal_class_persubj"].to_numpy(dtype=int)
X_w = per_subject_zscore(df, ALL)
groups_w = df["participant"].to_numpy()
print(f"window: X={X_w.shape}, subjects={len(np.unique(groups_w))}")

# Block-level
block = build_block_level_dataset(df)
block_feat_cols = [c for c in block.columns if c not in
                   {"participant", "ssq_post_total", "ssq_pre_total", "ssq_delta", "ssq_high"}]
X_b = block[block_feat_cols].to_numpy(dtype=float)
y_b = block["ssq_high"].to_numpy(dtype=int)
groups_b = block["participant"].to_numpy()
print(f"block: X={X_b.shape}, subjects={len(np.unique(groups_b))}")


shape: (15104, 157), features: 143


window: X=(15104, 143), subjects=32
block: X=(32, 288), subjects=32


## 2. XGBoost block — Optuna nested CV

20 trial × внутренний 3-fold по train-субъектам × внешний LOO (32 фолда) ≈ 30-50 мин CPU.

In [3]:
def objective_xgb_block(trial, X_tr, y_tr, groups_tr):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 400, step=50),
        "max_depth": trial.suggest_int("max_depth", 2, 6),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 10.0, log=True),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
    }
    gkf = GroupKFold(n_splits=3)
    f1s = []
    for tr_in, va_in in gkf.split(X_tr, y_tr, groups_tr):
        m = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", XGBClassifier(
                **params, objective="binary:logistic", eval_metric="logloss",
                tree_method="hist", n_jobs=-1, random_state=SEED, verbosity=0,
            )),
        ])
        m.fit(X_tr[tr_in], y_tr[tr_in])
        f1s.append(f1_score(y_tr[va_in], m.predict(X_tr[va_in]), average="macro"))
    return float(np.mean(f1s))

XGB_OUT = R / "exp08_xgb_block_nested.csv"
if XGB_OUT.exists():
    print(f"{XGB_OUT.name} уже существует — пропускаем")
    xgb_results = pd.read_csv(XGB_OUT)
else:
    from sklearn.model_selection import LeaveOneOut
    rows = []
    t_global = time.time()
    for fi, (tr, te) in enumerate(LeaveOneOut().split(X_b)):
        sampler = optuna.samplers.TPESampler(seed=SEED, n_startup_trials=5)
        study = optuna.create_study(direction="maximize", sampler=sampler)
        study.optimize(
            lambda t: objective_xgb_block(t, X_b[tr], y_b[tr], groups_b[tr]),
            n_trials=20, show_progress_bar=False,
        )
        best = study.best_params
        m = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", XGBClassifier(
                **best, objective="binary:logistic", eval_metric="logloss",
                tree_method="hist", n_jobs=-1, random_state=SEED, verbosity=0,
            )),
        ])
        m.fit(X_b[tr], y_b[tr])
        y_pred = m.predict(X_b[te])
        y_proba = m.predict_proba(X_b[te])[:, 1]
        rows.append({
            "fold": fi, "participant": str(groups_b[te][0]),
            "y_true": int(y_b[te][0]), "y_pred": int(y_pred[0]), "y_proba": float(y_proba[0]),
            "best_inner_f1": float(study.best_value),
            **{f"hp_{k}": v for k, v in best.items()},
        })
        if (fi + 1) % 4 == 0:
            print(f"  XGB block fold {fi+1}/32  ({time.time()-t_global:.0f}s)")
    xgb_results = pd.DataFrame(rows)
    xgb_results.to_csv(XGB_OUT, index=False)
    print(f"\nsaved {XGB_OUT.name}  total {time.time()-t_global:.0f}s")

# Aggregate metrics over LOO
y_true = xgb_results.y_true.to_numpy()
y_pred = xgb_results.y_pred.to_numpy()
y_proba = xgb_results.y_proba.to_numpy()
mt = compute_classification_metrics(y_true, y_pred, y_proba)
print(f"\nXGBoost block (Optuna nested LOO):  acc={mt['accuracy']:.4f}  f1={mt['f1_macro']:.4f}  auc={mt['auc_roc']:.4f}")
print("Сравнение: дефолтный XGBoost block — acc=0.7188, f1=0.7046")

print("\nЧастоты выбранных значений (top features):")
for c in [c for c in xgb_results.columns if c.startswith("hp_")]:
    if xgb_results[c].dtype.kind in "if":
        print(f"  {c}: median={xgb_results[c].median():.4f}  mean={xgb_results[c].mean():.4f}")
    else:
        print(f"  {c}: {dict(xgb_results[c].value_counts().head(3))}")


  XGB block fold 4/32  (11s)


  XGB block fold 8/32  (22s)


  XGB block fold 12/32  (32s)


  XGB block fold 16/32  (41s)


  XGB block fold 20/32  (51s)


  XGB block fold 24/32  (61s)


  XGB block fold 28/32  (72s)


  XGB block fold 32/32  (82s)

saved exp08_xgb_block_nested.csv  total 82s

XGBoost block (Optuna nested LOO):  acc=0.6250  f1=0.6000  auc=0.6944
Сравнение: дефолтный XGBoost block — acc=0.7188, f1=0.7046

Частоты выбранных значений (top features):
  hp_n_estimators: median=200.0000  mean=200.0000
  hp_max_depth: median=6.0000  mean=5.8438
  hp_learning_rate: median=0.2522  mean=0.2058
  hp_subsample: median=0.9729  mean=0.9441
  hp_colsample_bytree: median=0.5780  mean=0.5846
  hp_reg_lambda: median=0.1195  mean=0.4404
  hp_min_child_weight: median=1.0000  mean=1.0938


## 3. Random Forest window — Optuna nested CV

20 trial × внутренний 3-fold × внешний LOSO (32 фолда) ≈ 40-70 мин CPU.

In [4]:
def objective_rf_window(trial, X_tr, y_tr, groups_tr):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500, step=50),
        "max_depth": trial.suggest_categorical("max_depth", [None, 5, 10, 15, 20]),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", 0.5]),
    }
    gkf = GroupKFold(n_splits=3)
    f1s = []
    for tr_in, va_in in gkf.split(X_tr, y_tr, groups_tr):
        m = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(**params, n_jobs=-1, random_state=SEED)),
        ])
        m.fit(X_tr[tr_in], y_tr[tr_in])
        f1s.append(f1_score(y_tr[va_in], m.predict(X_tr[va_in]), average="macro"))
    return float(np.mean(f1s))

RF_OUT = R / "exp08_rf_window_nested.csv"
if RF_OUT.exists():
    print(f"{RF_OUT.name} уже существует — пропускаем")
    rf_results = pd.read_csv(RF_OUT)
else:
    rows = []
    t_global = time.time()
    splits = list(subject_independent_splits(df))
    for fi, (tr, te) in enumerate(splits):
        sampler = optuna.samplers.TPESampler(seed=SEED, n_startup_trials=5)
        study = optuna.create_study(direction="maximize", sampler=sampler)
        study.optimize(
            lambda t: objective_rf_window(t, X_w[tr], y_w[tr], groups_w[tr]),
            n_trials=20, show_progress_bar=False,
        )
        best = study.best_params
        m = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(**best, n_jobs=-1, random_state=SEED)),
        ])
        m.fit(X_w[tr], y_w[tr])
        y_pred = m.predict(X_w[te])
        y_proba = m.predict_proba(X_w[te])
        mt = compute_classification_metrics(y_w[te], y_pred, y_proba)
        test_pid = str(groups_w[te][0])
        row = {
            "fold": fi, "participant": test_pid,
            "best_inner_f1": float(study.best_value),
            **mt,
            **{f"hp_{k}": v for k, v in best.items()},
        }
        rows.append(row)
        if (fi + 1) % 4 == 0:
            print(f"  RF window fold {fi+1}/32 {test_pid}  acc={mt['accuracy']:.4f}  f1={mt['f1_macro']:.4f}  ({time.time()-t_global:.0f}s)")
    rf_results = pd.DataFrame(rows)
    rf_results.to_csv(RF_OUT, index=False)
    print(f"\nsaved {RF_OUT.name}  total {time.time()-t_global:.0f}s")

print(f"\nRF window (Optuna nested LOSO):")
print(f"  acc mean/std = {rf_results.accuracy.mean():.4f} ± {rf_results.accuracy.std():.4f}")
print(f"  f1  mean/std = {rf_results.f1_macro.mean():.4f} ± {rf_results.f1_macro.std():.4f}")
print(f"  auc mean/std = {rf_results.auc_roc.mean():.4f} ± {rf_results.auc_roc.std():.4f}")
print("Сравнение: дефолтный RF window — acc=0.6526, f1=0.6189")


  RF window fold 4/32 P4  acc=0.7203  f1=0.6819  (383s)


  RF window fold 8/32 P8  acc=0.6059  f1=0.5874  (776s)


  RF window fold 12/32 P12  acc=0.7373  f1=0.7086  (1260s)


  RF window fold 16/32 P16  acc=0.5466  f1=0.4320  (1799s)


  RF window fold 20/32 P20  acc=0.6356  f1=0.6205  (2176s)


  RF window fold 24/32 P24  acc=0.7119  f1=0.6983  (2583s)


  RF window fold 28/32 P28  acc=0.5678  f1=0.5346  (2954s)


  RF window fold 32/32 P32  acc=0.6610  f1=0.6538  (3345s)

saved exp08_rf_window_nested.csv  total 3345s

RF window (Optuna nested LOSO):
  acc mean/std = 0.6590 ± 0.0651
  f1  mean/std = 0.6249 ± 0.0669
  auc mean/std = 0.7025 ± 0.0833
Сравнение: дефолтный RF window — acc=0.6526, f1=0.6189


## 4. LSTM window — Optuna nested CV (GPU)

15 trial × внутренний 2-fold × внешний LOSO (32 фолда) ≈ 60-100 мин на GPU.

Меньше внутренних split-ов из-за стоимости обучения LSTM.

In [5]:
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {DEVICE}")

class LSTMTuned(nn.Module):
    def __init__(self, n_features, hidden, n_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden, num_layers=n_layers, batch_first=True,
                            dropout=dropout if n_layers > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden, 2)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(self.dropout(out[:, -1, :]))

def train_lstm_tuned(X_tr, y_tr, X_te, y_te, hp, n_features, epochs):
    set_seed(SEED)
    m = LSTMTuned(n_features, hp["hidden"], hp["n_layers"], hp["dropout"]).to(DEVICE)
    opt = torch.optim.Adam(m.parameters(), lr=hp["lr"])
    loss_fn = nn.CrossEntropyLoss()
    ds = TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr))
    loader = DataLoader(ds, batch_size=hp["batch"], shuffle=True, num_workers=0)
    m.train()
    for _ in range(epochs):
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = loss_fn(m(xb), yb)
            loss.backward()
            opt.step()
    m.eval()
    with torch.no_grad():
        Xt = torch.from_numpy(X_te).to(DEVICE)
        logits = m(Xt).cpu()
        probs = torch.softmax(logits, dim=1).numpy()
        preds = logits.argmax(dim=1).numpy()
    return preds, probs

X_w_f = X_w.astype(np.float32)
X_w_f = np.nan_to_num(X_w_f, nan=0.0, posinf=0.0, neginf=0.0)
seq_X, seq_y, seq_pid = build_sequences(df, X_w_f, y_w.astype(np.int64), k=K_DEFAULT)
print(f"sequences: {seq_X.shape}")

def objective_lstm(trial, seq_X_tr, seq_y_tr, seq_pid_tr):
    hp = {
        "hidden": trial.suggest_categorical("hidden", [64, 128, 256]),
        "n_layers": trial.suggest_int("n_layers", 1, 2),
        "dropout": trial.suggest_float("dropout", 0.1, 0.5),
        "lr": trial.suggest_float("lr", 1e-4, 3e-3, log=True),
        "batch": trial.suggest_categorical("batch", [128, 256]),
    }
    epochs = trial.suggest_int("epochs", 5, 12)
    gkf = GroupKFold(n_splits=2)
    f1s = []
    for tr_in, va_in in gkf.split(seq_X_tr, seq_y_tr, seq_pid_tr):
        if len(np.unique(seq_y_tr[tr_in])) < 2:
            continue
        preds, _ = train_lstm_tuned(seq_X_tr[tr_in], seq_y_tr[tr_in],
                                     seq_X_tr[va_in], seq_y_tr[va_in],
                                     hp, seq_X.shape[2], epochs)
        f1s.append(f1_score(seq_y_tr[va_in], preds, average="macro"))
    return float(np.mean(f1s)) if f1s else 0.0

LSTM_OUT = R / "exp08_lstm_window_nested.csv"
if LSTM_OUT.exists():
    print(f"{LSTM_OUT.name} уже существует — пропускаем")
    lstm_results = pd.read_csv(LSTM_OUT)
else:
    rows = []
    t_global = time.time()
    participants = np.unique(seq_pid)
    for fi, pid in enumerate(participants):
        te = (seq_pid == pid)
        tr = ~te
        if len(np.unique(seq_y[tr])) < 2 or te.sum() == 0:
            continue
        sampler = optuna.samplers.TPESampler(seed=SEED, n_startup_trials=4)
        study = optuna.create_study(direction="maximize", sampler=sampler)
        study.optimize(
            lambda t: objective_lstm(t, seq_X[tr], seq_y[tr], seq_pid[tr]),
            n_trials=15, show_progress_bar=False,
        )
        best = study.best_params
        epochs = best.pop("epochs")
        preds, probs = train_lstm_tuned(seq_X[tr], seq_y[tr], seq_X[te], seq_y[te],
                                         best, seq_X.shape[2], epochs)
        mt = compute_classification_metrics(seq_y[te], preds, probs)
        rows.append({
            "fold": fi, "participant": str(pid),
            "best_inner_f1": float(study.best_value),
            "hp_epochs": epochs,
            **mt,
            **{f"hp_{k}": v for k, v in best.items()},
        })
        if (fi + 1) % 4 == 0:
            print(f"  LSTM fold {fi+1}/32 {pid}  acc={mt['accuracy']:.4f}  f1={mt['f1_macro']:.4f}  ({time.time()-t_global:.0f}s)")
    lstm_results = pd.DataFrame(rows)
    lstm_results.to_csv(LSTM_OUT, index=False)
    print(f"\nsaved {LSTM_OUT.name}  total {time.time()-t_global:.0f}s")

print(f"\nLSTM window (Optuna nested LOSO):")
print(f"  acc mean/std = {lstm_results.accuracy.mean():.4f} ± {lstm_results.accuracy.std():.4f}")
print(f"  f1  mean/std = {lstm_results.f1_macro.mean():.4f} ± {lstm_results.f1_macro.std():.4f}")
print(f"  auc mean/std = {lstm_results.auc_roc.mean():.4f} ± {lstm_results.auc_roc.std():.4f}")
print("Сравнение: дефолтный LSTM — acc=0.6115, f1=0.6007")


device: cpu
sequences: (12800, 10, 143)


  LSTM fold 4/32 P12  acc=0.6775  f1=0.6599  (527s)


  LSTM fold 8/32 P16  acc=0.7375  f1=0.7294  (1132s)


  LSTM fold 12/32 P2  acc=0.7150  f1=0.6802  (1723s)


  LSTM fold 16/32 P23  acc=0.6350  f1=0.6277  (2399s)


  LSTM fold 20/32 P27  acc=0.5400  f1=0.5131  (2975s)


  LSTM fold 24/32 P30  acc=0.7350  f1=0.7068  (3559s)


  LSTM fold 28/32 P5  acc=0.4975  f1=0.4952  (4216s)


  LSTM fold 32/32 P9  acc=0.5850  f1=0.5761  (4819s)

saved exp08_lstm_window_nested.csv  total 4819s

LSTM window (Optuna nested LOSO):
  acc mean/std = 0.6379 ± 0.0878
  f1  mean/std = 0.6282 ± 0.0863
  auc mean/std = 0.6965 ± 0.1089
Сравнение: дефолтный LSTM — acc=0.6115, f1=0.6007


## 5. Сводка для диплома

In [6]:
# Default metrics (from prior experiments)
DEFAULT = {
    "XGBoost block":   {"acc": 0.7188, "f1": 0.7046, "auc": 0.7698},
    "RF window":       {"acc": 0.6526, "f1": 0.6189, "auc": 0.6945},
    "LSTM window":     {"acc": 0.6115, "f1": 0.6007, "auc": 0.6563},
}

# Tuned metrics
y_true_xgb = xgb_results.y_true.to_numpy()
y_pred_xgb = xgb_results.y_pred.to_numpy()
y_proba_xgb = xgb_results.y_proba.to_numpy()
mt_xgb = compute_classification_metrics(y_true_xgb, y_pred_xgb, y_proba_xgb)

TUNED = {
    "XGBoost block":   {"acc": float(mt_xgb["accuracy"]), "f1": float(mt_xgb["f1_macro"]), "auc": float(mt_xgb["auc_roc"])},
    "RF window":       {"acc": float(rf_results.accuracy.mean()), "f1": float(rf_results.f1_macro.mean()), "auc": float(rf_results.auc_roc.mean())},
    "LSTM window":     {"acc": float(lstm_results.accuracy.mean()), "f1": float(lstm_results.f1_macro.mean()), "auc": float(lstm_results.auc_roc.mean())},
}

# Best HP per model
def best_hp_summary(df, prefix="hp_"):
    cols = [c for c in df.columns if c.startswith(prefix)]
    summary = {}
    for c in cols:
        if df[c].dtype.kind in "if":
            summary[c[3:]] = f"median={df[c].median()}  range=[{df[c].min()}..{df[c].max()}]"
        else:
            top = df[c].value_counts().head(1)
            summary[c[3:]] = f"most common={top.index[0]} ({top.iloc[0]}/{len(df)})"
    return summary

print("=" * 80)
print("ИТОГ: Default vs Tuned (nested CV)")
print("=" * 80)
for name in DEFAULT:
    d, t = DEFAULT[name], TUNED[name]
    print(f"\n{name}:")
    for m in ["acc", "f1", "auc"]:
        delta = t[m] - d[m]
        sign = "+" if delta >= 0 else ""
        print(f"  {m:>4}: default={d[m]:.4f}  tuned={t[m]:.4f}  Δ={sign}{delta:+.4f}")

print("\n" + "=" * 80)
print("ЛУЧШИЕ ГИПЕРПАРАМЕТРЫ (median по 32 LOO/LOSO фолдам)")
print("=" * 80)
print("\nXGBoost block:")
for k, v in best_hp_summary(xgb_results).items():
    print(f"  {k}: {v}")
print("\nRandom Forest window:")
for k, v in best_hp_summary(rf_results).items():
    print(f"  {k}: {v}")
print("\nLSTM window:")
for k, v in best_hp_summary(lstm_results).items():
    print(f"  {k}: {v}")

# Save summary as JSON for OPTUNA_RESULTS.md
summary = {
    "default": DEFAULT, "tuned": TUNED,
    "best_hp": {
        "XGBoost block": best_hp_summary(xgb_results),
        "RF window": best_hp_summary(rf_results),
        "LSTM window": best_hp_summary(lstm_results),
    },
}
with open(R / "exp08_optuna_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print("\nsaved exp08_optuna_summary.json")


ИТОГ: Default vs Tuned (nested CV)

XGBoost block:
   acc: default=0.7188  tuned=0.6250  Δ=-0.0938
    f1: default=0.7046  tuned=0.6000  Δ=-0.1046
   auc: default=0.7698  tuned=0.6944  Δ=-0.0754

RF window:
   acc: default=0.6526  tuned=0.6590  Δ=++0.0064
    f1: default=0.6189  tuned=0.6249  Δ=++0.0060
   auc: default=0.6945  tuned=0.7025  Δ=++0.0080

LSTM window:
   acc: default=0.6115  tuned=0.6379  Δ=++0.0264
    f1: default=0.6007  tuned=0.6282  Δ=++0.0275
   auc: default=0.6563  tuned=0.6965  Δ=++0.0402

ЛУЧШИЕ ГИПЕРПАРАМЕТРЫ (median по 32 LOO/LOSO фолдам)

XGBoost block:
  n_estimators: median=200.0  range=[50..400]
  max_depth: median=6.0  range=[3..6]
  learning_rate: median=0.2521511680920944  range=[0.06504856968981275..0.298193695107655]
  subsample: median=0.9729161367647149  range=[0.8394633936788146..0.996951716289459]
  colsample_bytree: median=0.5780093202212182  range=[0.5076838686640521..0.8285726521637807]
  reg_lambda: median=0.11952270129143856  range=[0.100192168